# Cleaning 03: Quarantine Labels and Export

Use this notebook to remove non-trainable rows, validate the final dataset, and write the export files.

In [ ]:
import pandas as pd
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from helpers.logger import log_step, save_log

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

STAGE2_PATH = Path('../data/interim/cleaning_stage2.csv')
PROCESSED_PATH = Path('../data/processed/merged_inspections_licenses_inner_clean.csv')
PROCESSED_PARQUET_PATH = Path('../data/processed/merged_inspections_licenses_inner_clean.parquet')
LOG_PATH = Path('../data/interim/cleaning_log.csv')
QUARANTINE_PATH = Path('../data/interim/quarantine.csv')
FINAL_LOG_PATH = Path('../data/interim/cleaning_final_log.csv')

df_clean = pd.read_csv(STAGE2_PATH)
print('Loaded stage 2 shape:', df_clean.shape)

In [ ]:
def profile(df: pd.DataFrame, label: str) -> pd.DataFrame:
    missing = df.isna().sum().sort_values(ascending=False)
    missing_pct = (missing / len(df) * 100).round(2)

    out = pd.DataFrame({
        'missing_count': missing,
        'missing_pct': missing_pct,
        'dtype': df.dtypes.astype(str)
    }).sort_values(['missing_count', 'dtype'], ascending=[False, True])

    print(f'=== {label} ===')
    print('shape:', df.shape)
    print('full duplicates:', int(df.duplicated().sum()))
    print('duplicate Inspection ID:', int(df.duplicated(subset=['Inspection ID']).sum()))
    return out

profile(df_clean, 'Stage 2 input profile').head(20)

In [ ]:
# Action 1: Quarantine rows without a trainable label
rows_before, cols_before = df_clean.shape

if 'Results' in df_clean.columns:
    results_clean = df_clean['Results'].astype('string').str.strip()
    quarantine_mask = results_clean.isna() | results_clean.eq('') | results_clean.isin(['Not Ready', 'Business Not Located'])
    quarantine_df = df_clean.loc[quarantine_mask].copy()
    if not quarantine_df.empty:
        quarantine_df['quarantine_reason'] = 'missing_or_non_trainable_result'
    df_clean = df_clean.loc[~quarantine_mask].copy()
    df_clean['Results'] = results_clean.loc[~quarantine_mask]
else:
    quarantine_df = pd.DataFrame()

log_step('quarantine_non_trainable_labels', rows_before, df_clean.shape[0], cols_before, df_clean.shape[1], note='Removed missing / Not Ready / Business Not Located labels from training set')
print('Quarantined rows:', len(quarantine_df))

In [ ]:
# Final validation snapshot
post_profile = profile(df_clean, 'Post-clean profile')

print('Risk missing:', int(df_clean['Risk'].isna().sum()) if 'Risk' in df_clean.columns else 'N/A')
print('Risk levels:', df_clean['Risk'].value_counts(dropna=False).to_dict() if 'Risk' in df_clean.columns else 'N/A')
print('Malformed Zip count:', int(df_clean['Zip'].dropna().str.match(r'^\d{5}$').eq(False).sum()) if 'Zip' in df_clean.columns else 'N/A')
print('Future LICENSE TERM START DATE count:', int(((df_clean['Inspection Date'].notna()) & (df_clean['LICENSE TERM START DATE'].notna()) & (df_clean['LICENSE TERM START DATE'] > df_clean['Inspection Date'])).sum()) if {'Inspection Date', 'LICENSE TERM START DATE'}.issubset(df_clean.columns) else 'N/A')
print('Violations recorded flag present:', 'violations_recorded' in df_clean.columns)
print('Has prior inspection flag present:', 'has_prior_inspection' in df_clean.columns)
print('License matched flag present:', 'license_matched' in df_clean.columns)
print('Duplicate Inspection ID:', int(df_clean.duplicated(subset=['Inspection ID']).sum()))

post_profile.head(20)

In [ ]:
# Save cleaned data, quarantine data, and logs
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
QUARANTINE_PATH.parent.mkdir(parents=True, exist_ok=True)
FINAL_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(PROCESSED_PATH, index=False)
df_clean.to_parquet(PROCESSED_PARQUET_PATH, index=False)
quarantine_df.to_csv(QUARANTINE_PATH, index=False)
save_log(LOG_PATH)
save_log(FINAL_LOG_PATH)

print('Saved cleaned data (csv) to:', PROCESSED_PATH)
print('Saved cleaned data (parquet) to:', PROCESSED_PARQUET_PATH)
print('Saved quarantine rows to:', QUARANTINE_PATH)
print('Saved cleaning log to:', LOG_PATH)
print('Saved final log copy to:', FINAL_LOG_PATH)